In [63]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from collections import defaultdict
from tqdm import tqdm

In [64]:
df_meta = pd.read_parquet("/export/data_ml4ds/NextProcurement/Junio_2025/pliegosPlace/red_data_outsiders_2024_conTitleCPV_chunks/part_0004.parquet")
df_meta = df_meta[["place_id", "title"]]

In [70]:
model= SentenceTransformer("multi-qa-mpnet-base-dot-v1")

In [71]:
path_extractions = "/export/data_ml4ds/NextProcurement/Junio_2025/pliegosPlace/pruebas/Salida"

models = [
    "athene_v2_72b",
    "qwen_3_32b",
    "llama3_3_70b_instruct_q5_K_M",
    "falcon3_10b_instruct_fp16",
    "gemma2_9b",
    "mixtral_instruct",
    "qwen3_8b",
    "qwen2_5_72b",
    "llama3_1_8b",
    "mixtral_8x22b"
]

In [66]:
model_name = "qwen3_8b"
path_model_extract = f"{path_extractions}/part_0004_{model_name}.parquet"

df = pd.read_parquet(path_model_extract)

print(df[df.place_id == 'https://contrataciondelestado.es/sindicacion/PlataformasAgregadasSinMenores/14126855'].extracted_objective.values.tolist())

print(df[df.place_id == 'https://contrataciondelestado.es/sindicacion/PlataformasAgregadasSinMenores/14481784'].extracted_objective.values.tolist())

print(df[df.place_id == 'https://contrataciondelestado.es/sindicacion/PlataformasAgregadasSinMenores/14610130'].extracted_objective.values.tolist())

['<think>\nOkay, let\'s tackle this query. The user wants me to extract the "OBJETO" of a Spanish public tender document. The instructions are clear: I need to find a literal fragment from the context that describes the actual service or activity to be contracted. If there\'s no such mention, I should respond with \'/\'. Also, if there are multiple mentions, I have to choose the one that describes the final purpose, not just formal aspects.\n\nLooking at the provided context, there\'s a lot of technical jargon and sections like "Objetos de cálculo" and "Superficie de cálculo". But these seem to be about calculation objects and surfaces, which are more about the technical aspects of the project rather than the service being contracted. \n\nThen there\'s a section under "Fases d\'execució" listing various construction phases like "Replanteig d\'alineacions i nivells", "Obertura de buits en el terreny", etc. These are steps in the execution process, not the main service. \n\nFurther down,

In [67]:
df[df.place_id == 'https://contrataciondelestado.es/sindicacion/PlataformasAgregadasSinMenores/14610130'].url.values

array([{'administrativo': 'https://contractaciopublica.cat/portal-api/descarrega-document/300295013/EB62818FB5682BA54EBAEB06FA784029', 'tecnico': 'https://contractaciopublica.cat/portal-api/descarrega-document/300295293/5C0A1FEF1FAF8B9FD4F5E39AE38EB7D6'}],
      dtype=object)

In [72]:
results = defaultdict(list)

for model_name in tqdm(models, desc="Processing models"):
    
    path_model_extract = f"{path_extractions}/part_0004_{model_name}.parquet"

    df = pd.read_parquet(path_model_extract)
    #df = df[["place_id", "title", "texto_tecnico","extracted_objective", "generated_objective"]]
    print(df.columns)

    # failures due to no extractable text
    no_texto_tecnico = df[df.texto_tecnico == "[ERROR: PDF sin texto extraíble (posiblemente escaneado)]"]
    print(f"No extractable text: {len(no_texto_tecnico)}")
    
    failures = df[~df.place_id.isin(no_texto_tecnico.place_id)]
    failures = failures[failures.extracted_objective.str.contains("ERROR:")]
    print(f"Model: {model_name}, Failures: {len(failures)}")
    
    df_check = df[~df.place_id.isin(no_texto_tecnico.place_id) & ~df.place_id.isin(failures.place_id)]
    print(f"Left to check: {len(df_check)}")
    
    nf = df[df.extracted_objective == "/"]
    print(f"Number of empty extracted_objective: {len(nf)}")
    
    # remove think if it is a thinking model
    # @TODO: Maybe other models need other kind of cleanup
    if "<think>" in df_check.iloc[0].extracted_objective:
        print("Cleaning up <think> tags in extracted_objective and generated_objective columns.")
        df["extracted_objective"] = df.extracted_objective.apply(lambda x: x.split("</think>")[-1].strip())
        df["generated_objective"] = df.generated_objective.apply(lambda x: x.split("</think>")[-1].strip())
        
    embeddings_eo = model.encode(df['extracted_objective'].tolist(), convert_to_tensor=True, batch_size=64, show_progress_bar=True)
    embeddings_go = model.encode(df['generated_objective'].tolist(), convert_to_tensor=True, batch_size=64, show_progress_bar=True)
    embeddings_title = model.encode(df['title'].tolist(), convert_to_tensor=True, batch_size=64, show_progress_bar=True)
    
    similarities_eo_title = util.cos_sim(embeddings_eo, embeddings_title).diagonal() 
    similarities_go_title = util.cos_sim(embeddings_go, embeddings_title).diagonal()
    df['similarity_extracted'] = similarities_eo_title.cpu().numpy()
    df['similarity_generated'] = similarities_go_title.cpu().numpy()
    
    results[model_name].append({
        "model": model_name,
        "num_failures": len(failures),
        "num_no_text": len(no_texto_tecnico),
        "num_left_to_check": len(df_check),
        "num_empty_extracted": len(nf),
        "nf": nf,
        "similarity_extracted_mean": df['similarity_extracted'].mean(),
        "similarity_generated_mean": df['similarity_generated'].mean(),
        "similarity_extracted_std": df['similarity_extracted'].std(),
        "similarity_generated_std": df['similarity_generated'].std(),
    })

Processing models:   0%|          | 0/10 [00:00<?, ?it/s]

Index(['place_id', 'title', 'url', 'id', 'resultado_tecnico', 'path_tecnico',
       'resultado_administrativo', 'path_administrativo', 'texto_tecnico',
       'texto_administrativo', 'extracted_objective', 'generated_objective'],
      dtype='object')
No extractable text: 6
Model: athene_v2_72b, Failures: 8
Left to check: 86
Number of empty extracted_objective: 29


Processing models:  10%|█         | 1/10 [00:02<00:21,  2.35s/it]

Index(['place_id', 'title', 'url', 'id', 'resultado_tecnico', 'path_tecnico',
       'resultado_administrativo', 'path_administrativo', 'texto_tecnico',
       'texto_administrativo', 'extracted_objective', 'generated_objective'],
      dtype='object')
No extractable text: 6
Model: qwen_3_32b, Failures: 8
Left to check: 86
Number of empty extracted_objective: 0
Cleaning up <think> tags in extracted_objective and generated_objective columns.


Processing models:  20%|██        | 2/10 [00:03<00:14,  1.77s/it]

...

Processing models:  80%|████████  | 8/10 [00:14<00:03,  1.88s/it]

Index(['place_id', 'title', 'url', 'id', 'resultado_tecnico', 'path_tecnico',
       'resultado_administrativo', 'path_administrativo', 'texto_tecnico',
       'texto_administrativo', 'extracted_objective', 'generated_objective'],
      dtype='object')
No extractable text: 6
Model: llama3_1_8b, Failures: 8
Left to check: 86
Number of empty extracted_objective: 0


Processing models:  90%|█████████ | 9/10 [00:17<00:01,  1.95s/it]

Index(['place_id', 'title', 'url', 'id', 'resultado_tecnico', 'path_tecnico',
       'resultado_administrativo', 'path_administrativo', 'texto_tecnico',
       'texto_administrativo', 'extracted_objective', 'generated_objective'],
      dtype='object')
No extractable text: 6
Model: mixtral_8x22b, Failures: 0
Left to check: 94
Number of empty extracted_objective: 11


Processing models: 100%|██████████| 10/10 [00:18<00:00,  1.87s/it]


In [73]:
# Get best models as maximum Similarity Extracted Mean and Similarity Generated Mean with minimum Number of Empty Extracted Objectives
best_models = sorted(
    results.items(),
    key=lambda x: (
        -x[1][0]['similarity_extracted_mean'],
        -x[1][0]['similarity_generated_mean'],
        x[1][0]['num_empty_extracted']
    )
)

print("\nBest models:")
for model_name, result in best_models:
    res = result[0]
    print(
        f"Model: {model_name}, "
        f"Similarity Extracted Mean: {res['similarity_extracted_mean']:.4f} "
        f"(±{res.get('similarity_extracted_std', 0):.4f}), "
        f"Similarity Generated Mean: {res['similarity_generated_mean']:.4f} "
        f"(±{res.get('similarity_generated_std', 0):.4f}), "
        f"Number of Empty Extracted Objectives: {res['num_empty_extracted']}"
    )



Best models:
Model: falcon3_10b_instruct_fp16, Similarity Extracted Mean: 0.6064 (±0.2286), Similarity Generated Mean: 0.5514 (±0.1861), Number of Empty Extracted Objectives: 0
Model: mixtral_8x22b, Similarity Extracted Mean: 0.5940 (±0.1822), Similarity Generated Mean: 0.5590 (±0.1202), Number of Empty Extracted Objectives: 11
Model: llama3_3_70b_instruct_q5_K_M, Similarity Extracted Mean: 0.5668 (±0.2154), Similarity Generated Mean: 0.5282 (±0.1569), Number of Empty Extracted Objectives: 3
Model: qwen_3_32b, Similarity Extracted Mean: 0.5571 (±0.2477), Similarity Generated Mean: 0.5458 (±0.1672), Number of Empty Extracted Objectives: 0
Model: llama3_1_8b, Similarity Extracted Mean: 0.5566 (±0.1987), Similarity Generated Mean: 0.5204 (±0.1553), Number of Empty Extracted Objectives: 0
Model: mixtral_instruct, Similarity Extracted Mean: 0.5528 (±0.1759), Similarity Generated Mean: 0.5054 (±0.1442), Number of Empty Extracted Objectives: 0
Model: gemma2_9b, Similarity Extracted Mean: 0.5

In [ ]:
# Non found in extracted_objective
print("\nNon found in extracted_objective:")
for model_name, result in best_models:
    if result[0]['num_empty_extracted'] > 0:
        print(f"Model: {model_name}, Number of Empty Extracted Objectives: {result[0]['num_empty_extracted']}")
        for id, el in result[0]['nf'].iterrows():
            print(f"Place ID: {el['place_id']}, Title: {el['title']}, Path tecnico: {el['path_tecnico']}")


Non found in extracted_objective:
Model: mixtral_8x22b, Number of Empty Extracted Objectives: 11
Place ID: https://contrataciondelestado.es/sindicacion/PlataformasAgregadasSinMenores/14451181, Title: Obres relatives al projecte executiu del centre de transformació i quadre de baixa tensió del Port Olímpic de Barcelona, Path tecnico: //export/data_ml4ds/NextProcurement/Junio_2025/pliegosPlace/pdfs_descargados/outsiders/2014_tecnico.pdf
Place ID: https://contrataciondelestado.es/sindicacion/PlataformasAgregadasSinMenores/14299094, Title: Mantenimiento, inspección, limpieza, tratamiento de hidrofugación, reparación, descontaminación, y seguimiento de la vida útil de los trajes de intervención ante incendios estructurales del personal del Servicio Municipal de Extinción de Incendios y Salvamento, Path tecnico: //export/data_ml4ds/NextProcurement/Junio_2025/pliegosPlace/pdfs_descargados/outsiders/2015_tecnico.pdf
Place ID: https://contrataciondelestado.es/sindicacion/PlataformasAgregadasSi

In [ ]:
path_extractions = "/export/data_ml4ds/NextProcurement/Junio_2025/pliegosPlace/pruebas/PruebasK_SegundaParte"

#1 -> python3 src/rag/objective_extractor.py --llm_model_type_gen mixtral:8x22b --mode_extractive_generative generative
#2 -> python3 src/rag/objective_extractor.py --llm_model_type_gen mixtral:8x22b --mode_extractive_generative generative --fusion_alpha -1
#3 -> python3 src/rag/objective_extractor.py --llm_model_type_ex falcon3:10b-instruct-fp16 --mode_extractive_generative extractive
#4 -> python3 src/rag/objective_extractor.py --llm_model_type_ex falcon3:10b-instruct-fp16 --mode_extractive_generative extractive --fusion_alpha -1
#5 -> python3 src/rag/objective_extractor.py --llm_model_type qwen3:8b --mode_extractive_generative both
#6 -> python3 src/rag/objective_extractor.py --llm_model_type qwen3:8b --mode_extractive_generative both --fusion_alpha -1
#7 -> python3 src/rag/objective_extractor.py --llm_model_type_gen mixtral:8x22b --mode_extractive_generative generative --fusion_alpha 0
#8 -> python3 src/rag/objective_extractor.py --llm_model_type_gen mixtral:8x22b --mode_extractive_generative generative --fusion_alpha 1
#9 -> python3 src/rag/objective_extractor.py --llm_model_type_ex falcon3:10b-instruct-fp16 --mode_extractive_generative extractive --fusion_alpha 0
#10 -> python3 src/rag/objective_extractor.py --llm_model_type_ex falcon3:10b-instruct-fp16 --mode_extractive_generative extractive --fusion_alpha 1
#11 -> python3 src/rag/objective_extractor.py --llm_model_type qwen3:8b --mode_extractive_generative both --fusion_alpha 0
#12 -> python3 src/rag/objective_extractor.py --llm_model_type qwen3:8b --mode_extractive_generative both --fusion_alpha 1

models = [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9",
    "10",
    "11",
    "12"
]

results = defaultdict(list)

for model_name in tqdm(models, desc="Processing models"):
    
    path_model_extract = f"{path_extractions}/{model_name}/part_0004.parquet"

    df = pd.read_parquet(path_model_extract)
    #df = df[["place_id", "title", "texto_tecnico","extracted_objective", "generated_objective"]]
    print(df.columns)

    if not ('extracted_objective' in df.columns) :    
        df['extracted_objective']='ERROR'
   
    if not ('generated_objective' in df.columns) :    
        df['generated_objective']='ERROR'

    # failures due to no extractable text
    no_texto_tecnico = df[df.texto_tecnico == "[ERROR: PDF sin texto extraíble (posiblemente escaneado)]"]
    print(f"No extractable text: {len(no_texto_tecnico)}")
    
    failures = df[~df.place_id.isin(no_texto_tecnico.place_id)]
    failures = failures[failures.extracted_objective.str.contains("ERROR:")]
    print(f"Model: {model_name}, Failures: {len(failures)}")
    
    df_check = df[~df.place_id.isin(no_texto_tecnico.place_id) & ~df.place_id.isin(failures.place_id)]
    print(f"Left to check: {len(df_check)}")
    
    nf = df[df.extracted_objective == "/"]
    print(f"Number of empty extracted_objective: {len(nf)}")
    
    # remove think if it is a thinking model
    # @TODO: Maybe other models need other kind of cleanup
    if "<think>" in df_check.iloc[0].extracted_objective or "<think>" in df_check.iloc[0].generated_objective:
        print("Cleaning up <think> tags in extracted_objective and generated_objective columns.")
        df["extracted_objective"] = df.extracted_objective.apply(lambda x: x.split("</think>")[-1].strip())
        df["generated_objective"] = df.generated_objective.apply(lambda x: x.split("</think>")[-1].strip())
        
    embeddings_eo = model.encode(df['extracted_objective'].tolist(), convert_to_tensor=True, batch_size=64, show_progress_bar=True)
    embeddings_go = model.encode(df['generated_objective'].tolist(), convert_to_tensor=True, batch_size=64, show_progress_bar=True)
    embeddings_title = model.encode(df['title'].tolist(), convert_to_tensor=True, batch_size=64, show_progress_bar=True)
    
    similarities_eo_title = util.cos_sim(embeddings_eo, embeddings_title).diagonal() 
    similarities_go_title = util.cos_sim(embeddings_go, embeddings_title).diagonal()
    df['similarity_extracted'] = similarities_eo_title.cpu().numpy()
    df['similarity_generated'] = similarities_go_title.cpu().numpy()
    
    results[model_name].append({
        "model": model_name,
        "num_failures": len(failures),
        "num_no_text": len(no_texto_tecnico),
        "num_left_to_check": len(df_check),
        "num_empty_extracted": len(nf),
        "nf": nf,
        "similarity_extracted_mean": df['similarity_extracted'].mean(),
        "similarity_generated_mean": df['similarity_generated'].mean(),
        "similarity_extracted_std": df['similarity_extracted'].std(),
        "similarity_generated_std": df['similarity_generated'].std(),
    })

In [ ]:
# Get best models as maximum Similarity Extracted Mean and Similarity Generated Mean with minimum Number of Empty Extracted Objectives
best_models = sorted(
    results.items(),
    key=lambda x: (
        -x[1][0]['similarity_extracted_mean'],
        -x[1][0]['similarity_generated_mean'],
        x[1][0]['num_empty_extracted']
    )
)

print("\nBest models:")
for model_name, result in best_models:
    res = result[0]
    print(
        f"Model: {model_name}, "
        f"Similarity Extracted Mean: {res['similarity_extracted_mean']:.4f} "
        f"(±{res.get('similarity_extracted_std', 0):.4f}), "
        f"Similarity Generated Mean: {res['similarity_generated_mean']:.4f} "
        f"(±{res.get('similarity_generated_std', 0):.4f}), "
        f"Number of Empty Extracted Objectives: {res['num_empty_extracted']}"
    )